# 🎯 AI Image Detection - Full Training Pipeline

**Competition:** TechJam - AI-Generated Image Detection Challenge  
**Deadline:** September 1, 2026  

## 📋 What This Does:
- Downloads all 3 competition datasets (CIFAKE, SID_Set, WildFake)
- Trains transform-aware multi-branch ensemble
- Produces competition-ready model

## ⏱️ Timeline:
- Dataset download: ~1 hour
- Training: ~12 hours  
- Total: ~13 hours

## 🚀 Steps:
1. **Enable GPU:** Runtime → Change runtime type → T4 GPU  
2. **Run all cells** in order (Ctrl+F9)  
3. **Keep browser tab open** (Colab needs tab active)
4. **Download checkpoints** at the end

In [ ]:
# Verify GPU is enabled
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("❌ GPU NOT ENABLED!")
    print("👉 Go to: Runtime → Change runtime type → GPU (T4)")
    print("Then re-run this cell.")

## 📦 Step 1: Clone Repository

In [ ]:
# Clone your repository (update with your GitHub URL)
!git clone https://github.com/YOUR_USERNAME/techjam_aigenimagedetector.git
%cd techjam_aigenimagedetector

# Or upload files manually if repo is private
# from google.colab import files
# !mkdir -p project
# %cd project

## 📚 Step 2: Install Dependencies

In [ ]:
!pip install -q timm opencv-python scikit-image kaggle huggingface-hub
print("✅ Dependencies installed!")

## 🔑 Step 3: Setup Kaggle Credentials

In [ ]:
from google.colab import files
import os

print("📤 Upload your kaggle.json file (from https://www.kaggle.com/settings/account):")
uploaded = files.upload()

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Test credentials
!kaggle datasets list --page-size 1
print("✅ Kaggle credentials configured!")

## 📥 Step 4: Download All Datasets (~1 hour)

Downloading 3 datasets in parallel for speed.

In [ ]:
import os
os.makedirs('data/raw', exist_ok=True)
%cd data/raw

print("=== Starting Dataset Downloads ===")
print("This will take ~30-60 minutes total...\n")

In [ ]:
# Dataset 1: CIFAKE (120k images, ~2 minutes)
print("📦 [1/3] Downloading CIFAKE dataset...")
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d CIFAKE
!rm cifake-real-and-ai-generated-synthetic-images.zip

# Count images
cifake_count = sum([len(files) for r, d, files in os.walk('CIFAKE')])
print(f"✅ CIFAKE downloaded: {cifake_count:,} images\n")

In [ ]:
# Dataset 2: SID_Set (~10-20 minutes)
print("📦 [2/3] Downloading SID_Set from HuggingFace...")
from huggingface_hub import snapshot_download

try:
    dataset_path = snapshot_download(
        repo_id='saberzl/SID_Set',
        repo_type='dataset',
        local_dir='SID_Set',
        local_dir_use_symlinks=False
    )
    sid_count = sum([len(files) for r, d, files in os.walk('SID_Set')])
    print(f"✅ SID_Set downloaded: {sid_count:,} images\n")
except Exception as e:
    print(f"⚠️ SID_Set download failed: {e}")
    print("Continuing with other datasets...\n")

In [ ]:
# Dataset 3: WildFake (~20-30 minutes)
print("📦 [3/3] Downloading WildFake from ModelScope...")

try:
    # Install ModelScope SDK
    !pip install -q modelscope
    
    from modelscope.msdatasets import MsDataset
    
    # Download WildFake dataset
    ds = MsDataset.load(
        'hy2628982280/WildFake',
        cache_dir='WildFake'
    )
    
    wildfake_count = sum([len(files) for r, d, files in os.walk('WildFake')])
    print(f"✅ WildFake downloaded: {wildfake_count:,} images\n")
except Exception as e:
    print(f"⚠️ WildFake download failed: {e}")
    print("Continuing without WildFake...\n")

print("\n✅ All dataset downloads complete!")

In [ ]:
# Return to project root
%cd ../..

## 🗂️ Step 5: Organize Data for Training

In [ ]:
print("🗂️ Organizing datasets into real/ and fake/ folders...")

!mkdir -p data/processed/real data/processed/fake

# Copy CIFAKE
print("Copying CIFAKE...")
!cp -r data/raw/CIFAKE/train/REAL/* data/processed/real/ 2>/dev/null || true
!cp -r data/raw/CIFAKE/train/FAKE/* data/processed/fake/ 2>/dev/null || true
!cp -r data/raw/CIFAKE/test/REAL/* data/processed/real/ 2>/dev/null || true
!cp -r data/raw/CIFAKE/test/FAKE/* data/processed/fake/ 2>/dev/null || true

# Copy SID_Set (structure may vary - adjust as needed)
print("Copying SID_Set...")
!find data/raw/SID_Set -name "*real*" -o -name "*Real*" -o -name "*REAL*" | xargs -I {} cp -r {} data/processed/real/ 2>/dev/null || true
!find data/raw/SID_Set -name "*fake*" -o -name "*Fake*" -o -name "*FAKE*" -o -name "*ai*" -o -name "*AI*" | xargs -I {} cp -r {} data/processed/fake/ 2>/dev/null || true

# Copy WildFake (structure may vary - adjust as needed)
print("Copying WildFake...")
!find data/raw/WildFake -name "*real*" -o -name "*Real*" -o -name "*REAL*" | xargs -I {} cp -r {} data/processed/real/ 2>/dev/null || true
!find data/raw/WildFake -name "*fake*" -o -name "*Fake*" -o -name "*FAKE*" -o -name "*ai*" -o -name "*AI*" | xargs -I {} cp -r {} data/processed/fake/ 2>/dev/null || true

print("\n✅ Data organization complete!")

In [ ]:
# Verify final counts
import os

real_files = [f for f in os.listdir('data/processed/real') if f.endswith(('.jpg', '.jpeg', '.png'))]
fake_files = [f for f in os.listdir('data/processed/fake') if f.endswith(('.jpg', '.jpeg', '.png'))]

real_count = len(real_files)
fake_count = len(fake_files)
total = real_count + fake_count

print("="*50)
print("📊 FINAL DATASET STATISTICS")
print("="*50)
print(f"✓ Real images:     {real_count:,}")
print(f"✓ Fake images:     {fake_count:,}")
print(f"✓ Total images:    {total:,}")
print(f"✓ Balance ratio:   {fake_count/real_count:.2f}:1 (fake:real)")
print("="*50)

if total < 100000:
    print("⚠️ WARNING: Less than 100k images. Some datasets may have failed to download.")
elif total < 200000:
    print("✅ Good dataset size for training!")
else:
    print("🎉 Excellent! Large dataset for robust training!")

## 🏋️ Step 6: Training Phase 1 - Frequency Detector (~2 hours)

In [ ]:
import time
start_time = time.time()

print("🚀 Starting Frequency Detector training...")
print("Expected duration: ~2 hours\n")

!python training/train_frequency.py

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Frequency Detector training complete in {elapsed:.1f} hours!")
print(f"Checkpoint saved: checkpoints/frequency_detector_best.pth")

## 🏋️ Step 7: Training Phase 2 - Spatial Detector (~8 hours)

In [ ]:
start_time = time.time()

print("🚀 Starting Spatial Detector training...")
print("Expected duration: ~8 hours")
print("This is the longest phase. Go sleep! 😴\n")

!python training/train_spatial.py

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Spatial Detector training complete in {elapsed:.1f} hours!")
print(f"Checkpoint saved: checkpoints/spatial_detector_best.pth")

## 🏋️ Step 8: Training Phase 3 - Fusion Model (~30 min)

In [ ]:
start_time = time.time()

print("🚀 Starting Fusion Model training...")
print("Expected duration: ~30 minutes")
print("Almost done! 🎉\n")

!python training/train_fusion.py

elapsed = (time.time() - start_time) / 60
print(f"\n✅ Fusion Model training complete in {elapsed:.1f} minutes!")
print(f"Checkpoint saved: checkpoints/fusion_model_best.pth")

## 📊 Step 9: Evaluate Performance

In [ ]:
print("📊 Evaluating model performance...\n")
!python evaluation/evaluate.py

In [ ]:
print("🔬 Testing robustness to transformations...\n")
!python evaluation/robustness_test.py

## 💾 Step 10: Download Trained Models

In [ ]:
# Check all checkpoints exist
import os

checkpoints = [
    'checkpoints/frequency_detector_best.pth',
    'checkpoints/spatial_detector_best.pth',
    'checkpoints/fusion_model_best.pth'
]

print("Checking trained models:\n")
all_exist = True
for cp in checkpoints:
    exists = os.path.exists(cp)
    size = os.path.getsize(cp) / (1024**2) if exists else 0
    status = "✅" if exists else "❌"
    print(f"{status} {cp}: {size:.1f} MB")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✅ All models trained successfully!")
else:
    print("\n⚠️ Some models are missing. Check training logs above.")

In [ ]:
# Zip all models for download
!zip -r trained_models.zip checkpoints/ inference.py data/transforms.py data/dataset.py models/ training/utils.py evaluation/

print("\n📦 Package created: trained_models.zip")
print(f"📊 Size: {os.path.getsize('trained_models.zip') / (1024**2):.1f} MB")

In [ ]:
# Download to your computer
from google.colab import files

print("⬇️ Downloading trained_models.zip to your computer...")
files.download('trained_models.zip')

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print("\n📝 Next steps:")
print("1. Extract trained_models.zip on your local machine")
print("2. Run inference: python inference.py --input_dir images/ --output results.json")
print("3. Submit results.json to competition")
print("\n🏆 Good luck with the TechJam competition!")